In [1]:
import json, re, sys, urllib.request


In [4]:
import sys
sys.path.append("../")   # path to folder containing the python file

from utils.load_gtf_cgc_dresden import *
from ProteinExpression.load_pr_data import *
from utils.util_functions import *

In [ ]:
#!/usr/bin/env python3
"""
oncotree2hpo.py: propose an HPO term for each OncoTree code, for manual review.

  python oncotree2hpo.py my_codes.txt > oncotree_hpo.tsv   # one code per line (or 1st column)
  python oncotree2hpo.py > all_oncotree_hpo.tsv            # every current OncoTree code
"""

ONCOTREE_URL = "https://oncotree.mskcc.org/api/tumorTypes?version=oncotree_latest_stable"
HPO_URL = ("https://github.com/obophenotype/human-phenotype-ontology/"
           "releases/latest/download/hp.json")
NEOPLASM = "HP:0002664"

STOP = {"of", "the", "and", "or", "nos", "with", "type", "a", "an", "in", "not",
        "otherwise", "specified", "primary", "other", "cell"}
CANON = {"cancer": "carcinoma", "tumour": "tumor", "neoplasms": "neoplasm",
         "pulmonary": "lung", "renal": "kidney", "hepatic": "liver",
         "gastric": "stomach", "colonic": "colon", "ovary": "ovarian",
         "uterus": "uterine", "cutaneous": "skin", "pancreas": "pancreatic",
         "esophagus": "esophageal", "oesophageal": "esophageal",
         "testis": "testicular", "mammary": "breast", "cerebral": "brain",
         "urothelial": "transitional", "leukaemia": "leukemia",
         "hematologic": "hematological", "haematological": "hematological"}
# A match made only of these words says nothing about the site
GENERIC = {"neoplasm", "tumor", "carcinoma", "adenocarcinoma", "sarcoma", "lymphoma",
           "leukemia", "malignant", "benign", "squamous", "adenoma", "neuroendocrine",
           "mixed", "germ", "undifferentiated", "poorly", "differentiated", "high",
           "low", "grade", "metastatic", "carcinoid", "acute", "chronic"}
# Organ-level fallback, by OncoTree tissue. Labels are resolved against the live HPO;
# the first one found wins, and a warning is printed if none is found.
TISSUE_FALLBACK = {
    "adrenal gland": ["Neoplasm of the adrenal gland"],
    "ampulla of vater": ["Neoplasm of the small intestine", "Neoplasm of the gastrointestinal tract"],
    "biliary tract": ["Biliary tract neoplasm", "Neoplasm of the gastrointestinal tract"],
    "bladder/urinary tract": ["Neoplasm of the urinary bladder", "Bladder neoplasm",
                              "Neoplasm of the genitourinary tract"],
    "bone": ["Neoplasm of the skeletal system", "Bone neoplasm"],
    "bowel": ["Neoplasm of the large intestine", "Neoplasm of the colon",
              "Neoplasm of the gastrointestinal tract"],
    "breast": ["Neoplasm of the breast"],
    "cervix": ["Neoplasm of the cervix uteri", "Uterine neoplasm"],
    "cns/brain": ["Neoplasm of the central nervous system", "Neoplasm of the nervous system"],
    "esophagus/stomach": ["Neoplasm of the gastrointestinal tract", "Neoplasm of the stomach",
                          "Esophageal neoplasm"],
    "eye": ["Neoplasm of the eye"],
    "head and neck": ["Neoplasm of head and neck"],
    "kidney": ["Renal neoplasm"],
    "liver": ["Neoplasm of the liver"],
    "lung": ["Neoplasm of the lung"],
    "lymphoid": ["Hematological neoplasm", "Hematopoietic neoplasm"],
    "myeloid": ["Hematological neoplasm", "Hematopoietic neoplasm"],
    "ovary/fallopian tube": ["Ovarian neoplasm"],
    "pancreas": ["Neoplasm of the pancreas"],
    "penis": ["Neoplasm of the male external genitalia", "Neoplasm of the genitourinary tract"],
    "peripheral nervous system": ["Neoplasm of the peripheral nervous system",
                                  "Neoplasm of the nervous system"],
    "peritoneum": ["Neoplasm of the peritoneum"],
    "pleura": ["Neoplasm of the pleura", "Neoplasm of the respiratory system"],
    "prostate": ["Neoplasm of the prostate"],
    "skin": ["Neoplasm of the skin"],
    "soft tissue": ["Soft tissue neoplasm", "Neoplasm of the soft tissue"],
    "testis": ["Testicular neoplasm"],
    "thymus": ["Neoplasm of the thymus", "Neoplasm of the mediastinum"],
    "thyroid": ["Neoplasm of the thyroid gland"],
    "uterus": ["Uterine neoplasm"],
    "vulva/vagina": ["Neoplasm of the female external genitalia",
                     "Neoplasm of the genitourinary tract"],
}


def tokens(text):
    out = set()
    text = text.lower().replace("'s", "").replace("\u2019s", "")
    for raw in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text):
        parts = raw.split("-")
        for w in ([raw] if parts[0] == "non" else parts):   # keep "non-small" whole
            w = CANON.get(w, w)
            if len(w) > 4 and w.endswith("s") and not w.endswith(("ss", "is", "us")):
                w = CANON.get(w[:-1], w[:-1])
            if w not in STOP:
                out.add(w)
    return frozenset(out)


def fetch_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "oncotree2hpo/1.0"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)


def hp_id(iri):
    return iri.rsplit("/", 1)[-1].replace("_", ":")


def load_hpo():
    g = fetch_json(HPO_URL)["graphs"][0]
    terms, nci2hp, children = {}, {}, {}
    for n in g["nodes"]:
        hid, meta, label = hp_id(n["id"]), n.get("meta", {}), n.get("lbl") or ""
        if not hid.startswith("HP:") or meta.get("deprecated") or label.startswith("obsolete"):
            continue
        syns = [(s["val"], 1 if s.get("pred") == "hasExactSynonym" else 2)
                for s in meta.get("synonyms", [])]
        terms[hid] = {"label": label, "names": [(label, 0)] + syns}
        for x in meta.get("xrefs", []):
            if x["val"].startswith("NCIT:"):
                nci2hp.setdefault(x["val"][5:], []).append(hid)
    for e in g.get("edges", []):
        if e["pred"] == "is_a":
            children.setdefault(hp_id(e["obj"]), []).append(hp_id(e["sub"]))
    neo, stack = set(), [NEOPLASM]
    while stack:
        t = stack.pop()
        if t not in neo:
            neo.add(t)
            stack.extend(children.get(t, []))
    return terms, nci2hp, neo


def load_oncotree():
    ot, hist = {}, {}
    for t in fetch_json(ONCOTREE_URL):
        refs = t.get("externalReferences") or {}
        ot[t["code"]] = {"name": t.get("name") or "", "tissue": t.get("tissue") or "",
                         "parent": t.get("parent"), "nci": refs.get("NCI") or []}
        for old in t.get("history") or []:
            hist[old] = t["code"]
    return ot, hist


class Matcher:
    def __init__(self, terms, nci2hp, neo):
        self.terms, self.nci2hp, self.neo = terms, nci2hp, neo
        self.exact, self.neo_names = {}, []
        for hid, t in terms.items():
            for text, rank in t["names"]:
                toks = tokens(text)
                if toks:
                    self.exact.setdefault(toks, []).append((rank, hid))
                    if hid in neo:
                        self.neo_names.append((toks, hid, rank))

    def generic(self, hid):
        return tokens(self.terms[hid]["label"]) <= GENERIC

    def lexical(self, name):
        real = tokens(name)
        implied = (frozenset({"carcinoma"}) - real) if "adenocarcinoma" in real else frozenset()
        exact = min(self.exact.get(real, []), default=None)
        best = {}
        for toks, hid, rank in self.neo_names:
            if toks <= real | implied:
                key = (len(toks & real) + 0.5 * len(toks & implied), -rank)
                if key > best.get(hid, (-1, 0)):
                    best[hid] = key
        ranked = sorted(best, key=best.get, reverse=True)
        return (exact[1] if exact else None), ranked, best

    def direct(self, tt):
        exact, ranked, best = self.lexical(tt["name"])
        if exact:
            return (exact, "name_exact", "high"), ranked
        xs = sorted({h for c in tt["nci"] for h in self.nci2hp.get(c, [])})
        if len(xs) == 1:
            return (xs[0], "ncit_xref", "high"), ranked
        if xs:
            pick = max(xs, key=lambda h: best.get(h, (0, 0)))
            return (pick, "ncit_xref_multi", "medium"), xs + ranked
        for h in ranked:
            if not self.generic(h):
                return (h, "name_partial", "medium"), ranked
        return None, ranked


def resolve_tissues(m):
    out = {}
    for tissue, labels in TISSUE_FALLBACK.items():
        for lab in labels:
            ids = [h for _, h in sorted(m.exact.get(tokens(lab), [])) if h in m.neo]
            if ids:
                out[tissue] = ids[0]
                break
        else:
            print(f"warning: no HPO term for tissue '{tissue}', using Neoplasm", file=sys.stderr)
            out[tissue] = NEOPLASM
    return out


def map_code(tt, ot, m, tissue_hp):
    tissue_term = tissue_hp.get(tt["tissue"].lower())
    hit, ranked = m.direct(tt)
    if hit is None:
        p, seen = tt["parent"], set()
        while p in ot and p != "TISSUE" and p not in seen:
            seen.add(p)
            phit, _ = m.direct(ot[p])
            if phit:
                hit = (phit[0], "via_parent:" + p, "medium" if phit[2] == "high" else "low")
                break
            p = ot[p]["parent"]
    if hit is None and ranked:
        hit = (ranked[0], "name_generic", "low")
    if hit is None:
        hit = (tissue_term, "tissue", "low") if tissue_term else (NEOPLASM, "fallback", "low")
    alts = [h for h in dict.fromkeys(ranked + [tissue_term]) if h and h != hit[0]][:3]
    return hit, alts


def main():
    ot, hist = load_oncotree()
    m = Matcher(*load_hpo())
    tissue_hp = resolve_tissues(m)
    unknown = {t["tissue"] for t in ot.values() if t["tissue"].lower() not in tissue_hp} - {"", "Other"}
    if unknown:
        print("warning: tissues without fallback: " + ", ".join(sorted(unknown)), file=sys.stderr)

    codes = sa["Oncotree Code"].unique()

    upper = {c.upper(): c for c in ot}
    label = lambda h: m.terms[h]["label"] if h in m.terms else ""
    print("\t".join(["input_code", "oncotree_code", "oncotree_name", "tissue", "hpo_id",
                     "hpo_label", "method", "confidence", "alternatives"]))
    rows = []

    for c in codes:
        cur = c if c in ot else hist.get(c) or upper.get(c.upper())
        if not cur:
            rows.append([c, "", "", "", "", "", "not_in_oncotree", "", ""])
            continue
        tt = ot[cur]
        (hid, method, conf), alts = map_code(tt, ot, m, tissue_hp)
        rows.append([c, cur, tt["name"], tt["tissue"], hid, label(hid), method, conf,
                     "; ".join(f"{a} {label(a)}" for a in alts)])
    df = pd.DataFrame(rows, columns=["MASTER_oncotree_code", "extracted_oncotree_code", "oncotree_name", "tissue",
                                 "hpo_id", "hpo_label", "method", "confidence", "alternatives"])
    return df

if __name__ == "__main__":
    df = main()

input_code	oncotree_code	oncotree_name	tissue	hpo_id	hpo_label	method	confidence	alternatives


In [11]:
sa[sa["pid"] == "YBNFFN"]["snv_vcf"].values


<ArrowStringArray>
['/omics/odcf/project/hipo/hipo_021/sequencing/whole_genome_sequencing/view-by-pid/H021-YBNFFN/snv_results/paired/metastasis_blood/results_SNVCallingWorkflow-1.2.166-1_v1_1_2021-02-10_11h57/snvs_H021-YBNFFN.vcf.gz']
Length: 1, dtype: str